# Spark en Anaconda / VS Code

In [13]:
import os, sys

os.environ["JAVA_HOME"] = r"C:\Users\RyanHz\anaconda3\envs\spark\Library"
os.environ["PATH"] = r"C:\Users\RyanHz\anaconda3\envs\spark\Library\bin;" + os.environ["PATH"]

os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(sys.executable)
print(os.environ["JAVA_HOME"])

c:\Users\RyanHz\anaconda3\envs\spark\python.exe
C:\Users\RyanHz\anaconda3\envs\spark\Library


In [14]:
# 1) Configurar Java correcto ANTES de importar Spark
# Esta celda busca un Java compatible y evita que Spark use Java 26.

import os
import sys
import re
import glob
import shutil
import subprocess
from pathlib import Path

CONDA_PREFIX = Path(sys.prefix)

def java_major(java_exe: Path):
    try:
        result = subprocess.run(
            [str(java_exe), "-version"],
            capture_output=True,
            text=True,
            timeout=10
        )
        output = (result.stderr or "") + (result.stdout or "")
        match = re.search(r'version "([^"]+)"', output)
        if not match:
            return None, output
        version = match.group(1)
        if version.startswith("1.8"):
            return 8, output
        return int(version.split(".")[0]), output
    except Exception as e:
        return None, str(e)

candidates = []

# 1. Java instalado dentro del entorno conda: conda install -c conda-forge openjdk=8
candidates.append(CONDA_PREFIX / "Library" / "bin" / "java.exe")
candidates.append(CONDA_PREFIX / "bin" / "java.exe")

# 2. JAVA_HOME actual, si existe
if os.environ.get("JAVA_HOME"):
    candidates.append(Path(os.environ["JAVA_HOME"]) / "bin" / "java.exe")

# 3. Java que encuentre Windows en PATH
java_path = shutil.which("java")
if java_path:
    candidates.append(Path(java_path))

# 4. Búsqueda común en Windows
for pattern in [
    r"C:\Program Files\Java\jdk*\bin\java.exe",
    r"C:\Program Files\Java\jre*\bin\java.exe",
    r"C:\Program Files\Eclipse Adoptium\jdk*\bin\java.exe",
    r"C:\Program Files\Microsoft\jdk*\bin\java.exe",
]:
    candidates.extend(Path(p) for p in glob.glob(pattern))

# Quitar duplicados conservando orden
unique_candidates = []
seen = set()
for p in candidates:
    p = Path(p)
    key = str(p).lower()
    if key not in seen:
        seen.add(key)
        unique_candidates.append(p)

compatible = []
checked = []
for java_exe in unique_candidates:
    if java_exe.exists():
        major, info = java_major(java_exe)
        checked.append((str(java_exe), major))
        if major in (8, 11, 17):
            compatible.append((java_exe, major, info))

if not compatible:
    print("Java revisados:")
    for path, major in checked:
        print(f"- {path} -> Java {major}")
    raise RuntimeError(
        "No encontré Java compatible para Spark 3.5.x.\n\n"
        "Solución recomendada en Anaconda Prompt:\n"
        "conda activate spark\n"
        "conda install -c conda-forge openjdk=8 -y\n\n"
        "Luego cierra VS Code y ábrelo desde Anaconda Prompt con:\n"
        "conda activate spark\n"
        "code ."
    )

# Preferir Java del entorno conda; si no existe, usar el primer Java compatible encontrado.
java_exe, major, info = compatible[0]
JAVA_HOME = java_exe.parent.parent

os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PATH"] = str(JAVA_HOME / "bin") + os.pathsep + os.environ.get("PATH", "")

# Evita que PySpark tome una instalación externa como C:\spark.
os.environ.pop("SPARK_HOME", None)

# Forzar que workers usen el mismo Python del kernel.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python del notebook:")
print(sys.executable)
print("\nJAVA_HOME corregido:")
print(os.environ["JAVA_HOME"])
print("\nJava usado por Spark:")
print(info)


Python del notebook:
c:\Users\RyanHz\anaconda3\envs\spark\python.exe

JAVA_HOME corregido:
c:\Users\RyanHz\anaconda3\envs\spark\Library

Java usado por Spark:
openjdk version "17.0.18" 2026-01-20 LTS
OpenJDK Runtime Environment Zulu17.64+17-CA (build 17.0.18+8-LTS)
OpenJDK 64-Bit Server VM Zulu17.64+17-CA (build 17.0.18+8-LTS, mixed mode, sharing)



In [15]:
# 2) Verificar PySpark del entorno
# Debe salir PySpark 3.5.6

import pyspark
import sys

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("PySpark path:", pyspark.__file__)

if not pyspark.__version__.startswith("3.5"):
    raise RuntimeError(
        "Este notebook espera PySpark 3.5.x.\n"
        "En Anaconda Prompt ejecuta:\n"
        "conda activate spark\n"
        "pip uninstall pyspark py4j -y\n"
        "pip install pyspark==3.5.6"
    )


Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
PySpark: 3.5.6
PySpark path: c:\Users\RyanHz\anaconda3\envs\spark\Lib\site-packages\pyspark\__init__.py


In [16]:
# 3) Crear sesión de Spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PythonPi")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark


Py4JError: An error occurred while calling None.org.apache.spark.sql.SparkSession. Trace:
py4j.Py4JException: Constructor org.apache.spark.sql.SparkSession([class org.apache.spark.SparkContext, class java.util.HashMap]) does not exist
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:180)
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:197)
	at py4j.Gateway.invoke(Gateway.java:237)
	at py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
	at py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:108)
	at java.base/java.lang.Thread.run(Thread.java:840)



In [17]:
# 4) Prueba simple: estimar Pi con Spark

from random import random
from operator import add

particiones = 4
n = 10000 * particiones

def f(_):
    x = random() * 2 - 1
    y = random() * 2 - 1
    return 1 if x * x + y * y <= 1 else 0

count = spark.sparkContext.parallelize(range(1, n + 1), particiones).map(f).reduce(add)
pi = 4.0 * count / n

print(f"Pi aproximado: {pi}")


NameError: name 'spark' is not defined

In [8]:
# 5) Prueba con DataFrame

data = [("Bryan", 24), ("Spark", 35), ("Python", 11)]
df = spark.createDataFrame(data, ["nombre", "valor"])
df.show()


+------+-----+
|nombre|valor|
+------+-----+
| Bryan|   24|
| Spark|   35|
|Python|   11|
+------+-----+



In [9]:
# 6) Cerrar Spark cuando termines
spark.stop()


# Practicas

In [15]:
path_archivo = '../../../Archivos-Analisis/files-tarea-m33/vgsales.csv'

# DF de Spark
df = spark.read.csv(path_archivo)

In [ ]:
type(df)

pyspark.sql.dataframe.DataFrame

In [ ]:
df.show(10)

+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
| _c0|                 _c1|     _c2| _c3|         _c4|      _c5|     _c6|     _c7|     _c8|        _c9|        _c10|
+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
|   1|          Wii Sports|     Wii|2006|      Sports| Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform| Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing| Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports| Nintendo|   15.75|   11.01|    3.28|       2.96|          33|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing| Nintendo|

In [16]:
# Le decimos que el csv ya cuenta con encabezados 
df = spark.read.csv(path_archivo, header=True)

In [17]:
df.show(5)

+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports| Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform| Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing| Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports| Nintendo|   15.75|   11.01|    3.28|       2.96|          33|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing| Nintendo|   11.27|    8.89|   10.22|          1|       31.37|
+----+--------------------+--------+----+------------+---------+

In [18]:
df.dtypes

[('Rank', 'string'),
 ('Name', 'string'),
 ('Platform', 'string'),
 ('Year', 'string'),
 ('Genre', 'string'),
 ('Publisher', 'string'),
 ('NA_Sales', 'string'),
 ('EU_Sales', 'string'),
 ('JP_Sales', 'string'),
 ('Other_Sales', 'string'),
 ('Global_Sales', 'string')]

In [19]:
# Cuenta el numero de lineas
df.count()

16598

In [20]:
# Cambiamos el tipo de la columna Global_Sales
from pyspark.sql.types import IntegerType, FloatType

df = df.withColumn('NA_Sales',df.Global_Sales.cast(FloatType()))
df = df.withColumn('EU_Sales',df.Global_Sales.cast(FloatType()))
df = df.withColumn('JP_Sales',df.Global_Sales.cast(FloatType()))
df = df.withColumn('Other_Sales',df.Global_Sales.cast(FloatType()))
df = df.withColumn('Global_Sales',df.Global_Sales.cast(FloatType()))

In [21]:
df.dtypes

[('Rank', 'string'),
 ('Name', 'string'),
 ('Platform', 'string'),
 ('Year', 'string'),
 ('Genre', 'string'),
 ('Publisher', 'string'),
 ('NA_Sales', 'float'),
 ('EU_Sales', 'float'),
 ('JP_Sales', 'float'),
 ('Other_Sales', 'float'),
 ('Global_Sales', 'float')]

In [22]:
# Imprime la informacion de columnas de una manera jerarquica
df.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: float (nullable = true)
 |-- EU_Sales: float (nullable = true)
 |-- JP_Sales: float (nullable = true)
 |-- Other_Sales: float (nullable = true)
 |-- Global_Sales: float (nullable = true)



# Ordenamiento de resultados

- Uso de .sort()

In [ ]:
# Se importa el objeto Functions en F1 para seleccionar la columna
import pyspark.sql.functions as F1

# Se ordena por una columna
df.sort(F1.col("Platform")).show(10)

+----+---------------+--------+----+--------+------------+--------+--------+--------+-----------+------------+
|Rank|           Name|Platform|Year|   Genre|   Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+---------------+--------+----+--------+------------+--------+--------+--------+-----------+------------+
| 545|Missile Command|    2600|1980| Shooter|       Atari|    2.76|    2.76|    2.76|       2.76|        2.76|
|1431|      Centipede|    2600|1981| Shooter|       Atari|    1.36|    1.36|    1.36|       1.36|        1.36|
| 608| Space Invaders|    2600| N/A| Shooter|       Atari|    2.53|    2.53|    2.53|       2.53|        2.53|
| 240|       Pitfall!|    2600|1981|Platform|  Activision|     4.5|     4.5|     4.5|        4.5|         4.5|
| 736|        Frogger|    2600|1981|  Action|Parker Bros.|     2.2|     2.2|     2.2|        2.2|         2.2|
|1108|    Ms. Pac-Man|    2600|1981|  Puzzle|       Atari|    1.65|    1.65|    1.65|       1.65|        1.65|
|

In [ ]:
# Ordenamiento por dos columnas
df.sort(F1.col("Platform"), F1.col('Year')).show(10)

+----+--------------------+--------+----+--------+----------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|   Genre| Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+--------+----------+--------+--------+--------+-----------+------------+
| 259|           Asteroids|    2600|1980| Shooter|     Atari|    4.31|    4.31|    4.31|       4.31|        4.31|
| 545|     Missile Command|    2600|1980| Shooter|     Atari|    2.76|    2.76|    2.76|       2.76|        2.76|
|1768|             Kaboom!|    2600|1980|    Misc|Activision|    1.15|    1.15|    1.15|       1.15|        1.15|
|1971|            Defender|    2600|1980|    Misc|     Atari|    1.05|    1.05|    1.05|       1.05|        1.05|
|2671|              Boxing|    2600|1980|Fighting|Activision|    0.77|    0.77|    0.77|       0.77|        0.77|
|4027|          Ice Hockey|    2600|1980|  Sports|Activision|    0.49|    0.49|    0.49|

In [ ]:
# Usando orderBy   
df.orderBy('Year').show(10)

+----+---------------+--------+----+--------+----------+--------+--------+--------+-----------+------------+
|Rank|           Name|Platform|Year|   Genre| Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+---------------+--------+----+--------+----------+--------+--------+--------+-----------+------------+
| 545|Missile Command|    2600|1980| Shooter|     Atari|    2.76|    2.76|    2.76|       2.76|        2.76|
| 259|      Asteroids|    2600|1980| Shooter|     Atari|    4.31|    4.31|    4.31|       4.31|        4.31|
|1768|        Kaboom!|    2600|1980|    Misc|Activision|    1.15|    1.15|    1.15|       1.15|        1.15|
|1971|       Defender|    2600|1980|    Misc|     Atari|    1.05|    1.05|    1.05|       1.05|        1.05|
|2671|         Boxing|    2600|1980|Fighting|Activision|    0.77|    0.77|    0.77|       0.77|        0.77|
|4027|     Ice Hockey|    2600|1980|  Sports|Activision|    0.49|    0.49|    0.49|       0.49|        0.49|
|5368|        Freew

In [ ]:
df.orderBy('Year', 'Genre').show(10)

+----+---------------+--------+----+--------+--------------------+--------+--------+--------+-----------+------------+
|Rank|           Name|Platform|Year|   Genre|           Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+---------------+--------+----+--------+--------------------+--------+--------+--------+-----------+------------+
|5368|        Freeway|    2600|1980|  Action|          Activision|    0.34|    0.34|    0.34|       0.34|        0.34|
|2671|         Boxing|    2600|1980|Fighting|          Activision|    0.77|    0.77|    0.77|       0.77|        0.77|
|1768|        Kaboom!|    2600|1980|    Misc|          Activision|    1.15|    1.15|    1.15|       1.15|        1.15|
|1971|       Defender|    2600|1980|    Misc|               Atari|    1.05|    1.05|    1.05|       1.05|        1.05|
|6319|         Bridge|    2600|1980|    Misc|          Activision|    0.27|    0.27|    0.27|       0.27|        0.27|
|6898|       Checkers|    2600|1980|    Misc|   

In [ ]:
# Orden descendiente
df.sort(F1.col("Platform"), F1.col('Year').desc()).show(10)

+----+--------------------+--------+----+---------+-----------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|    Genre|  Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+---------+-----------+--------+--------+--------+-----------+------------+
|4380|Maze Craze: A Gam...|    2600| N/A|   Action|      Atari|    0.45|    0.45|    0.45|       0.45|        0.45|
|4471|      Super Breakout|    2600| N/A|   Puzzle|      Atari|    0.44|    0.44|    0.44|       0.44|        0.44|
|5063|             Hangman|    2600| N/A|   Puzzle|      Atari|    0.38|    0.38|    0.38|       0.38|        0.38|
|1515|           Adventure|    2600| N/A|Adventure|      Atari|     1.3|     1.3|     1.3|        1.3|         1.3|
|5659|            Dragster|    2600| N/A|   Racing| Activision|    0.32|    0.32|    0.32|       0.32|        0.32|
|2115|      Air-Sea Battle|    2600| N/A|  Shooter|      Atari|    0.98|

In [ ]:
# Ascendente y Descendiente
df.sort(F1.col("Platform").desc(), F1.col('Year').asc()).show(10)

+----+--------------------+--------+----+-------+--------------------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|  Genre|           Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+-------+--------------------+--------+--------+--------+-----------+------------+
|1433|   Ryse: Son of Rome|    XOne|2013| Action|Microsoft Game St...|    1.36|    1.36|    1.36|       1.36|        1.36|
|7822|         NBA Live 14|    XOne|2013| Sports|     Electronic Arts|    0.19|    0.19|    0.19|       0.19|        0.19|
|1700|             FIFA 14|    XOne|2013| Sports|     Electronic Arts|    1.19|    1.19|    1.19|       1.19|        1.19|
| 733|Assassin's Creed ...|    XOne|2013| Action|             Ubisoft|    2.21|    2.21|    2.21|       2.21|        2.21|
|2009|LEGO Marvel Super...|    XOne|2013| Action|Warner Bros. Inte...|    1.04|    1.04|    1.04|       1.04|        1.04|
| 858|       Bat

# Agrupacion de Resultados
- uso de groupBy

In [ ]:
df.groupBy('Platform').sum('Global_Sales').show()

+--------+--------------------+
|Platform|   sum(Global_Sales)|
+--------+--------------------+
|     3DO| 0.09999999776482582|
|      PC|  258.81999943964183|
|     PS3|   957.8399979360402|
|     NES|   251.0700025446713|
|      PS|   730.6599993146956|
|      DC|   15.97000003606081|
|     GEN|  28.360000303015113|
|     PS2|  1255.6399968694896|
|     3DS|  247.45999994501472|
|    PCFX|0.029999999329447746|
|      GG| 0.03999999910593033|
|    WiiU|   81.86000025086105|
|    SNES|  200.05000124499202|
|      GB|   255.4500018171966|
|     SCD|  1.8700000010430813|
|     N64|   218.8800003696233|
|     PS4|   278.0999997109175|
|     PSP|  296.27999946288764|
|    2600|   97.07999984174967|
|    XOne|  141.06000107899308|
+--------+--------------------+
only showing top 20 rows



In [23]:
df.groupBy(['Genre', 'Platform']).sum('Global_Sales').show()

+------------+--------+-------------------+
|       Genre|Platform|  sum(Global_Sales)|
+------------+--------+-------------------+
|        Misc|    2600|  3.579999938607216|
|      Sports|     SAT|  2.790000006556511|
|Role-Playing|     GEN|0.26999999955296516|
|    Platform|      GB|  54.90999959409237|
|      Puzzle|     NES| 20.999999906867743|
|        Misc|     PS2| 101.13999996334314|
|Role-Playing|     NES| 18.779999658465385|
|      Racing|     PSP| 34.729999953880906|
|      Puzzle|      PC| 0.9199999887496233|
|   Adventure|     PS3| 22.899999843910336|
|   Adventure|     PSV|  4.179999981075525|
|      Action|     PS2|  272.7599990274757|
|Role-Playing|      PC|  47.78000031597912|
|      Racing|      XB|   31.4899996612221|
|    Strategy|    WiiU| 1.2400000132620335|
|Role-Playing|     Wii|  14.05999998562038|
|      Racing|      GC|  21.88999987952411|
|    Fighting|      PS|  72.67999962344766|
|      Racing|     GBA|  18.79999982006848|
|  Simulation|     PS4| 0.770000

# Uso de Sql y tablas temporales con spark.sql

In [24]:
# Tabla temporal
df.createOrReplaceTempView("EMP")
# Comando SQL
sql_str = 'select Genre, Name, Rank, Publisher, NA_Sales, Global_Sales from EMP limit 20'
# Ejecucion
spark.sql(sql_str).show()

+------------+--------------------+----+--------------------+--------+------------+
|       Genre|                Name|Rank|           Publisher|NA_Sales|Global_Sales|
+------------+--------------------+----+--------------------+--------+------------+
|      Sports|          Wii Sports|   1|            Nintendo|   82.74|       82.74|
|    Platform|   Super Mario Bros.|   2|            Nintendo|   40.24|       40.24|
|      Racing|      Mario Kart Wii|   3|            Nintendo|   35.82|       35.82|
|      Sports|   Wii Sports Resort|   4|            Nintendo|    33.0|        33.0|
|Role-Playing|Pokemon Red/Pokem...|   5|            Nintendo|   31.37|       31.37|
|      Puzzle|              Tetris|   6|            Nintendo|   30.26|       30.26|
|    Platform|New Super Mario B...|   7|            Nintendo|   30.01|       30.01|
|        Misc|            Wii Play|   8|            Nintendo|   29.02|       29.02|
|    Platform|New Super Mario B...|   9|            Nintendo|   28.62|      

In [ ]:
sql_str = 'select Genre, sum(NA_Sales), sum(Global_Sales) from EMP group by Genre'
spark.sql(sql_str).show()

+------------+------------------+------------------+
|       Genre|     sum(NA_Sales)| sum(Global_Sales)|
+------------+------------------+------------------+
|   Adventure| 239.0399997998029| 239.0399997998029|
|      Sports|1330.9299955032766|1330.9299955032766|
|      Racing| 732.0399981327355| 732.0399981327355|
|Role-Playing| 927.3700027912855| 927.3700027912855|
|     Shooter|1037.3699989020824|1037.3699989020824|
|        Misc|  809.959999890998|  809.959999890998|
|    Platform| 831.3700044620782| 831.3700044620782|
|      Puzzle|244.95000079646707|244.95000079646707|
|    Fighting|448.90999998152256|448.90999998152256|
|      Action|1751.1799969691783|1751.1799969691783|
|    Strategy|175.11999987624586|175.11999987624586|
|  Simulation|392.20000046119094|392.20000046119094|
+------------+------------------+------------------+



: 

In [ ]:
sql_str = 'select Genre, Publisher, sum(NA_Sales), sum(Global_Sales) from EMP group by Genre, Publisher order by Genre, Publisher desc'
spark.sql(sql_str).show()